# 06- Merge

**Purpose:** The notebook that reads from every previous processed output. Use `geo_utils.sample_raster_at_points` to extract each conditioning factor's value at every landslide/non-landslide point from 05_labels. 

In [1]:
# Cell 1 — imports
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.append("../src")
from landslide_lk import geo_utils

DATA_PROCESSED = Path("../data/processed")

In [ ]:
#  reload labels 
labels_path = DATA_PROCESSED / "labels.parquet"
assert labels_path.exists(), "labels.parquet missing — run notebook 05 first"

labels = gpd.read_parquet(labels_path)
print(f"{len(labels)} points loaded: {labels['label'].sum()} landslide, {(labels['label']==0).sum()} non-landslide")

78 points loaded: 39 landslide, 39 non-landslide


In [3]:
# sample every available raster layer at every point
raster_layers = {
    "slope": "slope.tif",
    "aspect": "aspect.tif",
    "twi": "twi.tif",
    "spi": "spi.tif",
    "sti": "sti.tif",
    "dist_to_river": "dist_to_river.tif",
    "dist_to_road": "dist_to_road.tif",
    "landcover": "landcover.tif",
}

for col_name, filename in raster_layers.items():
    path = DATA_PROCESSED / filename
    if path.exists():
        labels[col_name] = geo_utils.sample_raster_at_points(path, labels)
        print(f"  {col_name}: sampled ({labels[col_name].notna().sum()}/{len(labels)} valid)")
    else:
        print(f"  {col_name}: MISSING at {path}, skipped")

  slope: sampled (78/78 valid)
  aspect: sampled (78/78 valid)
  twi: sampled (78/78 valid)
  spi: sampled (78/78 valid)
  sti: sampled (78/78 valid)
  dist_to_river: sampled (78/78 valid)
  dist_to_road: sampled (78/78 valid)
  landcover: sampled (78/78 valid)


In [4]:
#join rainfall (district-level)
rainfall_path = DATA_PROCESSED / "rainfall_static_district.parquet"
if rainfall_path.exists():
    static_rainfall = pd.read_parquet(rainfall_path)
    labels = labels.merge(static_rainfall, on="adm2_pcode", how="left")
    print(f"Rainfall joined: {labels['rainfall_dekad_avg_mm'].notna().sum()}/{len(labels)} matched")
else:
    print("rainfall_static_district.parquet MISSING, skipped")

Rainfall joined: 78/78 matched


In [5]:
# check for missing/null values before saving
print(labels.isnull().sum())
print(f"\nComplete rows (no missing values): {labels.dropna().shape[0]} / {len(labels)}")

geometry                 0
label                    0
source                   0
adm2_name                0
adm2_pcode               0
slope                    0
aspect                   0
twi                      0
spi                      0
sti                      0
dist_to_river            0
dist_to_road             0
landcover                0
rainfall_dekad_avg_mm    0
dtype: int64

Complete rows (no missing values): 78 / 78


In [6]:
# save the final training table
out_path = DATA_PROCESSED / "training_table.parquet"
labels.to_parquet(out_path)

print(f"Saved: {out_path}")
print(f"Shape: {labels.shape}")
labels.head()

Saved: ..\data\processed\training_table.parquet
Shape: (78, 14)


,geometry,label,source,adm2_name,adm2_pcode,slope,aspect,twi,spi,sti,dist_to_river,dist_to_road,landcover,rainfall_dekad_avg_mm
0,POINT (482851.481 753809.105),1,COOLR,Badulla,LK81,8.794240,9.317299,1.866234,0.154705,0.812484,253.139816,30.697712,10,57.518724
1,POINT (502684.208 768940.165),1,COOLR,Badulla,LK81,6.548500,217.219971,3.774061,0.573966,1.056619,0.000000,61.395424,10,57.518724
2,POINT (454860.312 776090.023),1,COOLR,Nuwara Eliya,LK23,14.880496,267.809479,3.404773,2.125720,3.663135,43.413120,86.826241,10,69.856326
3,POINT (406718.317 721439.442),1,COOLR,Kalutara,LK13,6.143241,205.529968,2.922179,0.215265,0.674252,43.413120,92.093132,10,92.672028
4,POINT (502309.571 747560.572),1,COOLR,Badulla,LK81,28.810682,161.704666,4.235428,20.899904,15.485044,0.000000,110.682175,30,57.518724


In [7]:
# non-spatial preview
labels.drop(columns="geometry").describe()

,label,slope,aspect,twi,spi,sti,dist_to_river,dist_to_road,landcover,rainfall_dekad_avg_mm
count,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000
mean,0.500000,14.534801,176.412109,3.353318,11.936753,3.219965,886.247070,274.093170,13.846154,69.576364
std,0.503236,10.106173,98.197205,2.515259,77.938423,3.020750,892.267334,416.471497,10.718447,11.083691
min,0.000000,0.001270,9.112791,0.231758,0.000022,0.000008,0.000000,0.000000,10.000000,57.518724
25%,0.000000,8.037387,84.597614,1.929319,0.319827,1.088457,217.033657,30.697712,10.000000,57.518724
50%,0.500000,12.779038,173.958015,2.809866,0.802838,2.110531,598.201172,89.459686,10.000000,66.415273
75%,1.000000,20.378850,250.827652,3.673835,1.890128,4.338801,1120.561951,370.061188,10.000000,76.637810
max,1.000000,54.127697,359.453186,17.102161,689.032043,15.485044,3933.621582,2306.417725,50.000000,92.672028


In [8]:
labels_csv = labels.copy()
labels_csv["longitude"] = labels_csv.geometry.to_crs("EPSG:4326").x
labels_csv["latitude"] = labels_csv.geometry.to_crs("EPSG:4326").y
labels_csv = labels_csv.drop(columns="geometry")

out_path = DATA_PROCESSED / "training_table.csv"
labels_csv.to_csv(out_path, index=False)
print(f"Saved: {out_path}, {labels_csv.shape[0]} rows, {labels_csv.shape[1]} columns")

Saved: ..\data\processed\training_table.csv, 78 rows, 15 columns


In [9]:
labels_csv.head()

,label,source,adm2_name,adm2_pcode,slope,aspect,twi,spi,sti,dist_to_river,dist_to_road,landcover,rainfall_dekad_avg_mm,longitude,latitude
0,1,COOLR,Badulla,LK81,8.794240,9.317299,1.866234,0.154705,0.812484,253.139816,30.697712,10,57.518724,80.8448,6.8196
1,1,COOLR,Badulla,LK81,6.548500,217.219971,3.774061,0.573966,1.056619,0.000000,61.395424,10,57.518724,81.0243,6.9565
2,1,COOLR,Nuwara Eliya,LK23,14.880496,267.809479,3.404773,2.125720,3.663135,43.413120,86.826241,10,69.856326,80.5913,7.0210
3,1,COOLR,Kalutara,LK13,6.143241,205.529968,2.922179,0.215265,0.674252,43.413120,92.093132,10,92.672028,80.1563,6.5261
4,1,COOLR,Badulla,LK81,28.810682,161.704666,4.235428,20.899904,15.485044,0.000000,110.682175,30,57.518724,81.0209,6.7631


## TODO

Start here — see the purpose note above and the corresponding section of `docs/data_sources.md`.